# 03b - Tester la branche spaCy

Ce notebook ne lit pas de PDF. Il teste uniquement la branche `spacy` du pipeline, avec les cas saisis dans `test_cases`.

`test_cases` contient seulement trois colonnes : `text`, `control_family`, `comment`.

Seule la colonne `text` est envoyee au traitement. `control_family` et `comment` servent uniquement a lire, filtrer et commenter les resultats dans le notebook.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
elif not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root


In [ ]:
from compliance_nlp import (
    load_generic_detection_rules,
    load_section_definitions,
    load_whitelist_terms,
)
from compliance_nlp.pipeline import analyze_text

import pandas as pd


## Parametres de branche


In [ ]:
BRANCH_NAME = "spacy"
ENABLED_BRANCHES = (BRANCH_NAME,)
MODEL_STORE_DIR = r"D:\Workspaces\modelStore"
SPACY_MODEL = rf"{MODEL_STORE_DIR}\fr_core_news_sm"
SPACY_SYNONYMS_PATH = ROOT / "configs" / "spacy_synonyms.csv"
BRANCH_SCORE_COLUMN = "spacy_score"

{
    "enabled_branches": ENABLED_BRANCHES,
    "model_store_dir": MODEL_STORE_DIR,
    "spacy_model": SPACY_MODEL if BRANCH_NAME == "spacy" else None,
    "spacy_synonyms_path": str(SPACY_SYNONYMS_PATH),
    "branch_score_column": BRANCH_SCORE_COLUMN,
}


## Chargement des referentiels


In [ ]:
sections = load_section_definitions(repo_root / "configs" / "sections.csv")
generic_rules = load_generic_detection_rules(repo_root / "configs" / "generic_detection_rules.csv")
whitelist_terms = load_whitelist_terms(repo_root / "configs" / "article9_whitelist.csv")

pd.DataFrame([
    {"referentiel": "sections", "count": len(sections)},
    {"referentiel": "central_detection_rules", "count": len(generic_rules)},
    {"referentiel": "central_general_rules", "count": sum(rule.rule_scope == "general" for rule in generic_rules)},
    {"referentiel": "central_article9_rules", "count": sum(rule.rule_scope == "article9" for rule in generic_rules)},
    {"referentiel": "article9_whitelist", "count": len(whitelist_terms)},
])


## Tableau des cas de test


In [ ]:
TEST_CASES_PATH = ROOT / "configs" / "test_cases.csv"
TEST_CASE_COLUMNS = ["text", "control_family", "comment"]

test_cases = pd.read_csv(TEST_CASES_PATH, keep_default_na=False)
missing_columns = set(TEST_CASE_COLUMNS) - set(test_cases.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans {TEST_CASES_PATH}: {sorted(missing_columns)}")

test_cases = test_cases[TEST_CASE_COLUMNS]
test_cases


## Fonctions d'execution


In [ ]:
def finding_key(finding):
    return finding.rule_id or finding.code


def set_to_text(values):
    return " | ".join(sorted(value for value in values if value))


def run_detector(text, case_id):
    return analyze_text(
        document_name=case_id,
        source_path="notebook",
        extracted_text=text,
        generic_rules=generic_rules,
        section_definitions=sections,
        whitelist_terms=whitelist_terms,
        enabled_branches=ENABLED_BRANCHES,
        spacy_model=SPACY_MODEL,
        spacy_synonyms_path=str(SPACY_SYNONYMS_PATH),
    )


def findings_to_rows(case, analysis):
    findings = analysis.findings
    if not findings:
        return [{
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "comment": case["comment"],
            "detection_engine": BRANCH_NAME,
            "predicted_key": None,
            "code": None,
            "rule_id": None,
            "rule_scope": None,
            "regulatory_family": None,
            "section": None,
            "category": None,
            "matched_term": None,
            "detection_type": None,
            "score": None,
            "branch_score": None,
            "generic_score": None,
            "spacy_score": None,
            "severity": None,
            "evidence": None,
        }]

    return [
        {
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "comment": case["comment"],
            "detection_engine": finding.detection_engine,
            "predicted_key": finding_key(finding),
            "code": finding.code,
            "rule_id": finding.rule_id,
            "rule_scope": finding.rule_scope,
            "regulatory_family": finding.regulatory_family,
            "section": finding.section,
            "category": finding.category,
            "matched_term": finding.matched_term,
            "detection_type": finding.detection_type,
            "score": finding.score,
            "branch_score": finding.branch_score,
            "generic_score": finding.generic_score,
            "spacy_score": finding.spacy_score,
            "severity": finding.severity,
            "evidence": finding.evidence,
        }
        for finding in findings
    ]


## Lancement du banc de test


In [ ]:
case_results = []
finding_rows = []

for case_index, case in test_cases.iterrows():
    case = case.copy()
    case["case_id"] = f"CASE_{case_index + 1:03d}"
    analysis = run_detector(case["text"], case["case_id"])
    findings = analysis.findings
    predicted_keys = {finding_key(finding) for finding in findings}
    predicted_categories = {finding.category for finding in findings if finding.category}
    predicted_detection_types = {finding.detection_type for finding in findings if finding.detection_type}
    branch_scores = [getattr(finding, BRANCH_SCORE_COLUMN) for finding in findings]
    max_branch_score = max([score for score in branch_scores if score is not None], default=None)

    case_results.append({
        "case_id": case["case_id"],
        "text": case["text"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detected": bool(findings),
        "predicted_keys": set_to_text(predicted_keys),
        "predicted_categories": set_to_text(predicted_categories),
        "predicted_detection_types": set_to_text(predicted_detection_types),
        "finding_count": len(findings),
        "max_branch_score": max_branch_score,
        "branch_errors": analysis.metadata.get("branch_errors"),
    })
    finding_rows.extend(findings_to_rows(case, analysis))

case_results_df = pd.DataFrame(case_results)
findings_df = pd.DataFrame(finding_rows)

case_results_df


## Synthese de detection


In [ ]:
pd.DataFrame([{
    "branch": BRANCH_NAME,
    "cases": len(case_results_df),
    "detected_cases": int(case_results_df["detected"].sum()),
    "undetected_cases": int((~case_results_df["detected"]).sum()),
    "finding_count": int(case_results_df["finding_count"].sum()),
    "avg_max_branch_score": case_results_df["max_branch_score"].mean(),
    "max_branch_score": case_results_df["max_branch_score"].max(),
}])


## Cas sans detection


In [ ]:
case_results_df[~case_results_df["detected"]][[
    "case_id",
    "control_family",
    "comment",
    "text",
    "branch_errors",
]]


## Detail des findings


In [ ]:
findings_df.sort_values(["case_id", "branch_score"], ascending=[True, False], na_position="last")


## Scores par type de detection


In [ ]:
findings_df[findings_df["predicted_key"].notna()].groupby(
    ["detection_engine", "control_family", "category", "detection_type"], dropna=False
).agg(
    count=("predicted_key", "count"),
    avg_score=("branch_score", "mean"),
    min_score=("branch_score", "min"),
    max_score=("branch_score", "max"),
).reset_index().sort_values(["detection_engine", "control_family", "category", "detection_type"])
